# fargv in Jupyter Notebooks

This notebook walks through the three ways to use **fargv** inside a Jupyter
kernel:

1. **Dict shortcut** — pass values directly, no CLI parsing.
2. **`jupyter` GUI mode** — auto-rendered ipywidgets form with an *Apply* button.
3. **Interactive mode** — `render_interactive` with *Start / Update / Reset / Kill / Help* buttons,
   designed for long-running cells where parameters may need live adjustment.

> **Requirement:** `pip install fargv ipywidgets`


## 1. Dict shortcut

Pass a `dict` as `given_parameters` to bypass all CLI / GUI logic and
set values directly.  Useful for reproducible experiments where every
parameter is fixed in the notebook cell.


In [1]:
import fargv

p, help_str = fargv.parse(
    {
        "learning_rate": 1e-3,
        "epochs":        10,
        "model":         ("resnet18", "resnet50", "vit_b16"),
        "data_dir":      "/data/imagenet",
        "output_dir":    "{data_dir}/runs",
        "dry_run":       False,
    },
    given_parameters={"learning_rate": 5e-4, "epochs": 20, "model": "resnet50"},
)
print(p)


namespace(learning_rate=0.0005, epochs=20, model='resnet50', data_dir='/data/imagenet', output_dir='/data/imagenet/runs', dry_run=False)


## 2. `jupyter` GUI mode

When `fargv.parse` detects it is running inside a Jupyter kernel it
automatically renders an ipywidgets form.  Click **Apply** to commit the
values and let `parse` return.

You can also request the GUI explicitly with `ui='jupyter'`.


In [2]:
# Run this cell — a widget form will appear below it.
# Edit the fields, then click Apply.

p, help_str = fargv.parse(
    {
        "learning_rate": 1e-3,
        "epochs":        10,
        "model":         ("resnet18", "resnet50", "vit_b16"),
        "data_dir":      "/data/imagenet",
        "output_dir":    "{data_dir}/runs",
        "dry_run":       False,
    },
    given_parameters=[],          # ignore Jupyter's own sys.argv
    ui='jupyter',
)
# Execution continues here after Apply is clicked:
print(p)


namespace(learning_rate=0.001, epochs=10, model='resnet18', data_dir='/data/imagenet', output_dir='/data/imagenet/runs', dry_run=False)


## 3. Interactive mode (`render_interactive`)

For long-running cells, use `render_interactive` from `fargv.ipywidgets`
directly.  The form blocks until **Start** is clicked; the cell then
continues running while the widget panel remains visible.

The buttons behave as follows:

| Button | Behaviour |
|--------|-----------|
| **Start** | Apply widget values → parameter dict, unblock the cell. |
| **Update** | Push changed values → dict mid-run (enabled only when dirty). |
| **Reset** | Revert widgets ← last committed values (enabled only when dirty). |
| **Kill** | Send `SIGTERM` — triggers `atexit` handlers before exit. |
| **Help** | Toggle the help-text panel. |


In [3]:
import time
from fargv.ipywidgets import render_interactive

params = {
    "learning_rate": 1e-3,
    "log_every":     50,
    "stop_early":    False,
}

# Blocks here — edit the fields and click Start.
render_interactive(params, help_str="learning_rate: step size\nlog_every: steps between log lines")

# Simulated training loop — Update button changes take effect each iteration.
for step in range(1, 201):
    time.sleep(0.05)   # replace with real work
    if step % params['log_every'] == 0:
        print(f"step {step:4d}  lr={params['learning_rate']:.2e}")
    if params['stop_early']:
        print(f"Early stop at step {step}")
        break


step   50  lr=1.00e-03
step  100  lr=1.00e-03
step  150  lr=1.00e-03
step  200  lr=1.00e-03


## 4. Saving and reloading parameters with `--auto_configure`

When `fargv.parse` is called with a real program name it injects
`--config` and `--auto_configure` automatically.  You can use
`dump_config` / `load_config` directly inside a notebook to persist
the current parameter set to a JSON file and reload it later.


In [ ]:
import json, pathlib
from fargv.config import dump_config, load_config

snapshot_path = pathlib.Path('experiment_params.json')

# Save current params
dump_config(params, snapshot_path)
print('Saved:', snapshot_path.read_text())

# Reload later
reloaded = load_config(snapshot_path)
print('Reloaded:', reloaded)


## 5. Displaying the help string

`fargv.parse` always returns `(namespace, help_str)`.  Render it in a
notebook cell for a quick reference of all parameters and their defaults.


In [ ]:
p, help_str = fargv.parse(
    {
        "learning_rate": 1e-3,
        "epochs":        10,
        "model":         ("resnet18", "resnet50", "vit_b16"),
        "data_dir":      "/data/imagenet",
    },
    given_parameters={},
    auto_define_config=False,
)
print(help_str)
